In [4]:
from sklearn.model_selection import train_test_split
from datetime import datetime
import pandas as pd

# --- Rutas de entrada ---
DATA_PATH     = r'C:\Users\AaronMCC\Documents\Rest-Mex2025\dataset\rest_mex_cleaned_v2.pkl'

# --- Rutas de salida ---

now = datetime.now() # Obtener la fecha y hora actual
date_time = now.strftime("%d_%m_%Y_%H_%M_%S")

OUTPUT_DIR        = r'C:\Users\AaronMCC\Documents\Rest-Mex2025\dataset' 

DATASET_TRAIN     = OUTPUT_DIR + r'\split_train_' + date_time + '.pkl'
DATASET_VAL       = OUTPUT_DIR + r'\split_val_' + date_time + '.pkl'
DATASET_TEST      = OUTPUT_DIR + r'\split_test_' + date_time + '.pkl'


# --- Division del dataset ---
TRAIN_SIZE  = 0.70
VAL_SIZE    = 0.15
TEST_SIZE   = 0.15
RANDOM_SEED = 42

# ==============================================================================
# 1. CARGA DE DATOS
# ==============================================================================
print("Cargando dataset...")
df = pd.read_pickle(DATA_PATH)
df['label'] = df['Polarity'].astype(int) - 1  # Rango 0-4
print(f"Total de ejemplos: {len(df):,}")

# ==============================================================================
# 2. DIVISION 70 / 15 / 15 ESTRATIFICADA
# ==============================================================================
train_df, temp_df = train_test_split(
    df,
    test_size=VAL_SIZE + TEST_SIZE,
    stratify=df['label'],
    random_state=RANDOM_SEED
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=TEST_SIZE / (VAL_SIZE + TEST_SIZE),
    stratify=temp_df['label'],
    random_state=RANDOM_SEED
)

print(f"Train: {len(train_df):,} ({100*len(train_df)/len(df):.1f}%)")
print(f"Val:   {len(val_df):,} ({100*len(val_df)/len(df):.1f}%)")
print(f"Test:  {len(test_df):,} ({100*len(test_df)/len(df):.1f}%)")

train_df.to_pickle(DATASET_TRAIN)
val_df.to_pickle(DATASET_VAL)
test_df.to_pickle(DATASET_TEST)

print(f"Split TRAIN guardado en: {DATASET_TRAIN}")
print(f"Split VALIDATION guardado en: {DATASET_VAL}")
print(f"Split TEST guardado en: {DATASET_TEST}")

Cargando dataset...
Total de ejemplos: 207,688
Train: 145,381 (70.0%)
Val:   31,153 (15.0%)
Test:  31,154 (15.0%)
Split TRAIN guardado en: C:\Users\AaronMCC\Documents\Rest-Mex2025\dataset\split_train_15_03_2026_03_28_20.pkl
Split VALIDATION guardado en: C:\Users\AaronMCC\Documents\Rest-Mex2025\dataset\split_val_15_03_2026_03_28_20.pkl
Split TEST guardado en: C:\Users\AaronMCC\Documents\Rest-Mex2025\dataset\split_test_15_03_2026_03_28_20.pkl


In [ ]:
# ── Celda 1: importar el pipeline ──────────────────────────────────────────
import importlib, augmentation_pipeline as aug
importlib.reload(aug)  

TRAIN_PKL      = r'C:\Users\AaronMCC\Documents\Rest-Mex2025\dataset\split_train_15_03_2026_03_28_20.pkl'  # CSV del split de TRAIN (post-limpieza)
OUTPUT_CSV     = r'C:\Users\AaronMCC\Documents\Rest-Mex2025\resultados\train_augmented.csv'     # CSV aumentado resultante
FASTTEXT_PATH  = r'C:\Users\AaronMCC\Documents\Rest-Mex2025\dataset\cc.es.300.bin'           # Modelo FastText (opcional; deja "" para omitir)


In [ ]:
# ── Celda 2: cargar el split de train ──────────────────────────────────────
import pandas as pd

df_train = pd.read_pickle(aug.TRAIN_PKL)
print(f"Train cargado: {len(df_train):,} filas")

In [ ]:
# ── Celda 3: dry run (SIEMPRE primero) ─────────────────────────────────────
aug.dry_run(df_train)

In [ ]:
# ── Celda 4: pipeline completo ─────────────────────────────────────────────
# FastText opcional — si tienes cc.es.300.bin en el mismo directorio
from pathlib import Path
ft_model = None
if Path(aug.FASTTEXT_PATH).exists():
    from gensim.models.fasttext import load_facebook_model
    ft_model = load_facebook_model(aug.FASTTEXT_PATH)

df_aumentado = aug.run_augmentation(
    df_train,
    output_path=aug.OUTPUT_CSV,
    ft_model=ft_model,
    skip_dry_run=True,   # ya hiciste el dry run en la celda anterior
)

In [ ]:
# ── Celda 5: verificar resultado ───────────────────────────────────────────
print(df_aumentado["label"].value_counts().sort_index())
print(f"\nTotal: {len(df_aumentado):,} filas")